In [ ]:
# Install dependencies if needed:
# !pip install ipywidgets matplotlib pandas ipydatagrid
%matplotlib widget
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from ipydatagrid import DataGrid
import pandas as pd

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv(r'..\Code\159_planets_all_columns_with_AH_2.csv')

# -----------------------------
# SETTINGS
# -----------------------------
plot_pairs = [('pl_Teq', 'A/H'), ('pl_Teq', 'pl_tsm'), ('pl_Teq', 'Feature Height (ppm)')]
plot_index = 0
selected_planets = pd.DataFrame(columns=df.columns)

# -----------------------------
# WIDGETS
# -----------------------------
x_dropdown = widgets.Dropdown(description='X-axis:')
y_dropdown = widgets.Dropdown(description='Y-axis:')
input_box = widgets.Text(description='Search:', placeholder='e.g., 800')
left_btn = widgets.Button(description='<')
right_btn = widgets.Button(description='>')
info_out = widgets.Output()
plot_out = widgets.Output()
clear_btn = widgets.Button(description='Clear Selection', button_style='warning')


# Scrollable table
grid = DataGrid(df, selection_mode='row', layout={'height': '300px'})
display(grid)

# -----------------------------
# FIGURE
# -----------------------------
#fig, ax = plt.subplots(figsize=(6,4))

# -----------------------------
# HELPER FUNCTIONS
# -----------------------------
def update_dropdowns():
    numeric_cols = list(df.select_dtypes(include='number').columns)
    x_dropdown.options = numeric_cols
    y_dropdown.options = numeric_cols
    
    # Safe defaults
    if x_dropdown.value is None:
        x_dropdown.value = 'pl_Teq' if 'pl_Teq' in numeric_cols else numeric_cols[0]
    if y_dropdown.value is None:
        y_dropdown.value = 'A/H' if 'A/H' in numeric_cols else numeric_cols[1] if len(numeric_cols) > 1 else numeric_cols[0]



def plot_scatter(x_col, y_col):
    if x_col not in df.columns or y_col not in df.columns:
        return

    with plot_out:
        plot_out.clear_output(wait=True)
        fig, ax = plt.subplots()
        ax.scatter(df[x_col], df[y_col], s=20, alpha=0.5, picker=True)  # picker=True here
        if not selected_planets.empty:
            ax.scatter(selected_planets[x_col], selected_planets[y_col], s=60, alpha=0.7, fc = 'None', ec='red', marker='o')
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.set_title("Planet comparison")
        ax.grid(True)
        fig.canvas.mpl_connect('pick_event', onpick_multi)  # connect here, to THIS fig
        plt.show()

# -----------------------------
# CALLBACKS
# -----------------------------
def onpick_multi(event):
    global selected_planets
    if event.mouseevent.button != 1:  # left click only
        return
    ind = event.ind[0]
    planet = df.iloc[ind]
    
    # toggle selection
    if planet['pl_name'] in selected_planets['pl_name'].values:
        selected_planets = selected_planets[selected_planets['pl_name'] != planet['pl_name']]
    else:
        selected_planets = pd.concat([selected_planets, pd.DataFrame([planet])], ignore_index=True)
    
    # redraw
    plot_scatter(x_dropdown.value, y_dropdown.value)
    
    # info output
    with info_out:
        clear_output(wait=True)
        display(f"Selected planets ({len(selected_planets)}):")
        if not selected_planets.empty:
            display(selected_planets[['pl_name', x_dropdown.value, y_dropdown.value]])

def on_input_change(change):
    global selected_planets
    if change['type'] == 'change' and change['name'] == 'value' and change['new']:
        try:
            val = float(change['new'])
        except:
            with info_out:
                clear_output(wait=True)
                display("Enter a numeric value")
            return
        
        # search within 5% of value
        matches = df[abs(df[x_dropdown.value] - val) < (0.05 * val)]
        if matches.empty:
            with info_out:
                clear_output(wait=True)
                display("No planets found in range")
            return
        
        # add matches
        for _, planet in matches.iterrows():
            if planet['pl_name'] not in selected_planets['pl_name'].values:
                selected_planets = pd.concat([selected_planets, pd.DataFrame([planet])], ignore_index=True)
        
        plot_scatter(x_dropdown.value, y_dropdown.value)
        
        with info_out:
            clear_output(wait=True)
            display(f"Selected planets ({len(selected_planets)}):")
            display(selected_planets[['pl_name', x_dropdown.value, y_dropdown.value]])

def on_left_click(b):
    global plot_index
    plot_index = (plot_index - 1) % len(plot_pairs)
    x_dropdown.value, y_dropdown.value = plot_pairs[plot_index]
    plot_scatter(x_dropdown.value, y_dropdown.value)

def on_right_click(b):
    global plot_index
    plot_index = (plot_index + 1) % len(plot_pairs)
    x_dropdown.value, y_dropdown.value = plot_pairs[plot_index]
    plot_scatter(x_dropdown.value, y_dropdown.value)


def on_clear_click(b):
    global selected_planets
    selected_planets = pd.DataFrame(columns=df.columns)
    
    plot_scatter(x_dropdown.value, y_dropdown.value)
    
    with info_out:
        clear_output(wait=True)
        display("Selection cleared.")

# -----------------------------
# CONNECT CALLBACKS
# -----------------------------

update_dropdowns()



input_box.observe(on_input_change)
left_btn.on_click(on_left_click)
right_btn.on_click(on_right_click)
clear_btn.on_click(on_clear_click)

def on_axis_change(change):
    plot_scatter(x_dropdown.value, y_dropdown.value)

x_dropdown.observe(on_axis_change, names='value')
y_dropdown.observe(on_axis_change, names='value')



# -----------------------------
# LAYOUT
# -----------------------------
control_box = widgets.HBox([left_btn, right_btn, x_dropdown, y_dropdown, input_box, clear_btn])
display(control_box, info_out, plot_out)
# -----------------------------
# INITIAL PLOT

# -----------------------------
update_dropdowns()
plot_scatter(x_dropdown.value, y_dropdown.value)
#plt.show()


DataGrid(auto_fit_params={'area': 'all', 'padding': 30, 'numCols': None}, corner_renderer=None, default_render…

Output()

Output()

In [29]:
selected_planets

,pl_name,pl_orbper,pl_orbsmax,pl_bmasse,pl_rade,pl_radelim,pl_radeerr1,pl_radeerr2,pl_trandep,pl_trandur,...,sy_jmagerr_avg,g_planet_cgs,g_planet_cgs_err,pl_Teq,pl_Teq_err,pl_tsm,pl_tsm_err,A/H,Feature Height (ppm),scale height (m)
0,Kepler-28 b,5.910000,0.05687,1.63,1.959,0.0,0.043,-0.042,0.106290,2.3254,...,0.023,417.089296,117.824887,678.102495,13.270195,19.250652,5.585480,16.085459,518.309084,275156.488631
1,K2-138 f,12.757580,0.10447,1.63,2.904,0.0,0.164,-0.111,0.085069,3.2000,...,0.021,189.803708,192.971475,679.019832,10.564121,123.786439,126.748291,23.679148,1473.362572,605468.045291
2,Kepler-83 b,9.770000,0.07295,2.94,2.830,0.0,0.410,-0.410,0.189780,2.4764,...,0.021,360.482986,139.164012,520.634849,17.048542,24.699918,12.710742,13.558124,664.302056,244434.174209
3,G 9-40 b,5.745998,0.04180,4.00,1.900,0.0,0.065,-0.065,0.365000,1.6930,...,0.022,1088.085698,186.845861,402.897029,9.281290,92.560901,18.492568,12.065977,413.523295,62667.796436
4,Kepler-138 c,13.781500,0.09130,2.30,1.510,0.0,0.040,-0.040,0.075600,2.2630,...,0.022,990.567908,242.618897,410.103782,7.636523,23.613467,6.089126,12.538517,122.156299,70068.526053
5,Kepler-83 c,20.090000,0.11796,2.03,2.360,0.0,0.350,-0.350,0.143210,3.5881,...,0.021,357.917015,133.063998,409.428906,13.404682,16.059401,8.173393,14.159017,458.218246,193601.901211
6,L 98-59 d,7.450729,0.04940,1.64,1.627,0.0,0.041,-0.041,0.144500,0.8400,...,0.027,608.385780,40.180842,380.657929,9.834607,327.812499,32.623315,13.582445,619.602811,105893.496142
7,TOI-244 b,7.397225,0.05590,2.68,1.520,0.0,0.120,-0.120,0.106000,1.7000,...,0.023,1139.089715,220.470155,418.004218,17.953292,65.272159,18.957898,12.299703,168.629099,62106.376933
8,TOI-4438 b,7.446280,0.05340,5.40,2.520,0.0,0.130,-0.130,0.404572,2.0210,...,0.021,835.031754,190.673056,398.372005,19.294676,123.703315,34.382190,12.053764,467.104373,80741.958999
9,TOI-776 c,15.665323,0.10010,6.90,2.047,0.0,0.081,-0.078,0.117700,2.9320,...,0.018,1617.051002,610.662779,384.070425,9.662281,40.430694,15.903126,11.771643,85.321447,40197.644956
